# Add buildings to Seattle topo

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
import pandas as pd
import shapely
from shapely import wkt
from shapely.geometry import box
from shapely.affinity import translate
from shapely import plotting as shapely_plotting
from clawpack.geoclaw import topotools, kmltools
from clawpack.visclaw import plottools
from clawpack.visclaw.colormaps import make_colormap
import folium

In [ ]:
csv_file = 'Seattle_Inventory.csv'
df = pd.read_csv(csv_file)

In [ ]:
df['geometry'] = df['geometry'].apply(wkt.loads)

In [ ]:
poly = df['geometry'][0]
type(poly)

In [ ]:
shapely_plotting.plot_polygon(poly, add_points=False, color='blue', alpha=0.5)
gca().set_aspect(1/cos(pi*47/180))

In [ ]:
#topo = topotools.Topography('/Users/rjl/git/ACS_2022/topo/topofiles/seattle_13s_mhw.asc',3)
#topo.crop(coarsen=3).plot()

## Load 1/9" tile

In [ ]:
from pathlib import Path
import numpy as np
from netCDF4 import Dataset

mhw_name = 'puget_sound_mhw_19_n47x75_w122x50_2023v1.nc'
mhw_path = Path('../examples/seattle_debris') / mhw_name

with Dataset(mhw_path) as ds:
    lon = np.asarray(ds['lon'][:])
    lat = np.asarray(ds['lat'][:])
    west, east, south, north = -122.41, -122.31, 47.57, 47.65
    i0 = max(0, np.searchsorted(lon, west) - 1)
    i1 = min(len(lon), np.searchsorted(lon, east) + 1)
    j0 = max(0, np.searchsorted(lat, south) - 1)
    j1 = min(len(lat), np.searchsorted(lat, north) + 1)
    x, y = lon[i0:i1], lat[j0:j1]
    Z = np.ma.filled(ds['z'][j0:j1, i0:i1], -500.).astype(float)

print(f'{mhw_path}: {Z.shape}, elevation [{Z.min():.1f}, {Z.max():.1f}] m MHW')

### Vertical datum

The NOAA tile is already in Mean High Water (MHW), so no NAVD88-to-MHW offset is applied. The longitude and latitude arrays increase from west to east and south to north, matching `Topography.set_xyZ`.

In [ ]:
topo19 = topotools.Topography()
topo19.set_xyZ(x, y, Z)

In [ ]:
topo19.crop(coarsen=9).plot()

## Plot all buildings from inventory

In [ ]:
fig,ax = subplots(figsize=(8,8))
#topo_sea19.plot(axes=ax, cb_kwargs={'shrink':0.5})

for k in range(len(df['geometry'])):
    poly = df['geometry'][k]
    try:
        vertices = list(poly.exterior.coords)
        xp = array([p[0] for p in vertices])
        yp = array([p[1] for p in vertices])
        ax.fill(xp,yp,'r')
    except:
        print(f'+++ skipping k = {k}')

ax.contour(topo19.X, topo19.Y, topo19.Z, [0], colors='b', linewidths=0.5)
ax.set_aspect(1/cos(pi*47/180))
ticklabel_format(useOffset=False)
ax.set_xlim(-122.4, -122.3)
ax.set_ylim(47.57,47.64)
grid(True)

## Smaller modeling region to start

In [ ]:
extent = [-122.36, -122.32, 47.58, 47.61]
topo_sea19 = topo19.crop(extent)

In [ ]:
fig,ax = subplots(figsize=(10,10))
topo_sea19.plot(axes=ax, cb_kwargs={'shrink':0.5})

for k in range(0,1000,100):
    poly = df['geometry'][k]
    vertices = list(poly.exterior.coords)
    xp = array([p[0] for p in vertices])
    yp = array([p[1] for p in vertices])
    ax.fill(xp,yp,'r')
    #shapely_plotting.plot_polygon(poly, add_points=False, color='red')

In [ ]:
#extent = [-122.345, -122.33, 47.588, 47.603]
extent = [-122.345, -122.325, 47.588, 47.605]
topo2 = topo_sea19.crop(extent)

In [ ]:
x1e,x2e,y1e,y2e = extent
buildings = []
    
for k in range(len(df['geometry'])):
    poly = df['geometry'][k]
    try:
        vertices = list(poly.exterior.coords)
        xp = array([p[0] for p in vertices])
        yp = array([p[1] for p in vertices])
        
        if (xp.min() < x2e) and (xp.max() > x1e) and \
           (yp.min() < y2e) and (yp.max() > y1e):
            if poly.area > 5e-8:
                bbox = [xp.min(), xp.max(), yp.min(), yp.max()]
                buildings.append((bbox, poly))              
    except:
        print(f'+++ skipping k = {k}, possibly multiple polygons')

print(f'Found {len(buildings)} buildings in extent {extent}')

In [ ]:
x1e,x2e,y1e,y2e = extent

In [ ]:
fig,ax = subplots(figsize=(8,8))
#topo_sea19.plot(axes=ax, cb_kwargs={'shrink':0.5})

for building in buildings:
    bbox, poly = building
    try:
        vertices = list(poly.exterior.coords)
        xp = array([p[0] for p in vertices])
        yp = array([p[1] for p in vertices])
        ax.fill(xp,yp,'r')
    except:
        print(f'+++ skipping k = {k}')

ax.contour(topo_sea19.X, topo_sea19.Y, topo_sea19.Z, [0], colors='b')
ax.plot([x1e,x2e,x2e,x1e,x1e],[y1e,y1e,y2e,y2e,y1e],'g')
ax.set_aspect(1/cos(pi*47/180))
ticklabel_format(useOffset=False)
grid(True)

## Need to upsample topo to get finer resolution?

Before adding buildings, need topo that is perhaps 1m resolution?

In [ ]:
topo2 = None
topo2 = topo_sea19.crop(extent)
Z = topo2.Z.copy()
Z0 = zeros(Z.shape) # a version with only buildings, no topo

n = 0
for i in range(len(topo2.x)):
    for j in range(len(topo2.y)):
        for k,building in enumerate(buildings):
            bbox, poly = building
            x1,x2,y1,y2 = bbox
            if (x1 <= topo2.x[i] <= x2) and (y1 <= topo2.y[j] <= y2):
                #print(f'x={topo2.x[i]:.5f}, y={topo2.y[j]:.5f} in bbox of building {k}')
                #import pdb; pdb.set_trace()
                if poly.contains(shapely.Point(topo2.x[i],topo2.y[j])):
                    n += 1
                    Z[j,i] += 20.
                    Z0[j,i] += 20.

print(f'Based on {len(buildings)} buildings, changed Z at {n} grid points')
topo3 = topotools.Topography()
topo3.set_xyZ(topo2.x, topo2.y, Z)

topo_only_buildings = topotools.Topography()
topo_only_buildings.set_xyZ(topo2.x, topo2.y, Z0)

In [ ]:
fig,ax = subplots(figsize=(8,8))
topo3.plot(axes=ax, cb_kwargs={'shrink':0.5})
ax.contour(topo3.X, topo3.Y, topo3.Z, [0,4,8,12], colors='r', linewidths=0.5)
grid(True)
title('Topography with buildings added');
fname = 'SeattleBuildingsTopo.png'
savefig(fname)
print('Created ',fname)

In [ ]:
fig,ax = subplots(figsize=(8,8))
topo_only_buildings.plot(axes=ax, cb_kwargs={'shrink':0.5})
ax.contour(topo_only_buildings.X, topo_only_buildings.Y, topo_only_buildings.Z, [0,4,8,12], colors='r', linewidths=0.5)
grid(True)
title('Topography of only buildings');

In [ ]:
from scipy.interpolate import RegularGridInterpolator

#i got AttributeErrors trying to use Topography.make_function(), so i think it got removed from clawpack, here's custom version
def make_topo_func(topo):
    interp = RegularGridInterpolator((topo.x, topo.y), topo.Z.T,method='linear', bounds_error=False, fill_value=None)
    return lambda x, y: interp(array(list(zip(x, y))))

topo2f = make_topo_func(topo2)
topo3f = make_topo_func(topo3)

In [ ]:
xtrans = linspace(-122.334, -122.330, 1000)
y0 = 47.5995
ytrans = y0 * ones(xtrans.shape)
xtransm = (xtrans - xtrans[0]) * 111e3*cos(pi*y0/180)  # in meters
plot(xtransm, topo3f(xtrans,ytrans), 'r')
plot(xtransm, topo2f(xtrans,ytrans), 'g')
grid(True)
xlabel('Distance (m)')
ylabel('Elevation (m)')
title(f'Transect at y = {y0}');

In [ ]:
xtrans = linspace(-122.334, -122.330, 1000)
y0 = 47.595
ytrans = y0 * ones(xtrans.shape)
xtransm = (xtrans - xtrans[0]) * 111e3*cos(pi*y0/180)  # in meters
plot(xtransm, topo3f(xtrans,ytrans), 'r')
plot(xtransm, topo2f(xtrans,ytrans), 'g')
grid(True)
xlabel('Distance (m)')
ylabel('Elevation (m)')
grid(True)
title(f'Transect at y = {y0}');

In [ ]:
fname = 'SeattleBuildings1_19s.asc'
topo3.write(fname, topo_type=3, header_style='arcgis', Z_format='%.3f')
print('Created ',fname)

In [ ]:
#!cp SeattleBuildings1_19s.asc /Users/rjl/git/ACS_2022/Runs/Seattle/SFL_buildings/

## Check that adjustment to MHW makes topo line up with old 1/3"

In [ ]:
topo_old = topotools.Topography('elliott_bay.tt3', topo_type=3) #change file path as needed

In [ ]:
#extent = [-122.36, -122.32, 47.58, 47.61]
#topo_sea19 = topo19.crop(extent)
extent_compare = [-122.38, -122.3, 47.56, 47.63]
topo_compare = topo_old.crop(extent_compare)
topof_compare = make_topo_func(topo_compare)

In [ ]:
xtrans = linspace(-122.36, -122.32, 1000)
y0 = 47.598
ytrans = y0 * ones(xtrans.shape)

figure(figsize=(9,6))
plot(xtrans, topof_compare(xtrans,ytrans), 'm')
plot(xtrans, topo2f(xtrans,ytrans), 'g')
grid(True)
xlabel('Longitude')
ylabel('Elevation (m)')
grid(True)
ylim(-5,10)
title(f'Transect at y = {y0}');

## Plot buildings on interactive map

In [ ]:
#tiles = 'OpenStreetMap'  # default tiles
tiles = 'OpenTopoMap' 
location = (47.596, -122.336)
m = folium.Map(location=location, tiles=tiles, zoom_start=14)
m

In [ ]:

bldgs = topo_only_buildings
png_filename = 'bldgs.png'

trans_red = make_colormap({0.:[1,0,0,0], 1.:[1,0,0,1]})
Z_footprints = minimum(bldgs.Z, 1.)  # 1 in footprints, 0 elsewhere
fig,ax,png_extent,kml_dpi = kmltools.pcolorcells_for_kml(bldgs.X, bldgs.Y, Z_footprints,
                                                 png_filename=png_filename,
                                                 dpc=2, cmap=trans_red)


In [ ]:
x1,x2,y1,y2 = png_extent
image_bounds = [[y1,x1], [y2,x2]]
xc = (x1+x2)/2
yc = (y1+y2)/2
image_bounds

In [ ]:
#tiles = 'OpenStreetMap'  # default tiles
tiles = 'OpenTopoMap' 
location = (47.596, -122.336)
m = folium.Map(location=location, tiles=tiles, zoom_start=14)

# Add the saved image as an overlay
folium.raster_layers.ImageOverlay(
    image=png_filename,
    bounds=image_bounds,
    opacity=0.8, # Adjust opacity as needed
    name='Buildings'
).add_to(m)

# Add a layer control to toggle the overlay
folium.LayerControl().add_to(m)
m